# GraphKT on Kaggle

Notebook này clone chính repo `linhnnh688/graph-ml` rồi chạy theo các module trong `src/gkt`.

Workflow: sửa code trong repo local → `git push` → trên Kaggle chạy lại cell clone (xóa + clone lại) là lấy code mới.

Yêu cầu: bật **GPU** (Runtime → Accelerator → GPU).

In [ ]:
%%capture
import os
import torch

os.environ["TORCH"] = torch.__version__.split("+")[0]
os.environ["PYTHONWARNINGS"] = "ignore"
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-geometric
!pip install -q tqdm scikit-learn

In [ ]:
# Clone lại repo để luôn lấy code mới nhất từ GitHub
import os
import shutil

REPO_URL = "https://github.com/linhnnh688/graph-ml.git"
REPO_DIR = "graph-ml"

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# Setup path + import module từ src/gkt
import sys

sys.path.insert(0, os.path.abspath(REPO_DIR))

import pandas as pd
import torch

from src.gkt.graph_net import GAT, GCN, GraphSAGE
from src.gkt.graphkt import init_models, train

assert torch.cuda.is_available(), "Bật GPU trước khi chạy (Accelerator → GPU)"
print("torch:", torch.__version__, "| cuda:", torch.cuda.get_device_name(0))

In [ ]:
# Hyperparameters
num_epochs = 15
batch_size = 8
block_size = 2048
skill_embd_dim = 128
graph_net = GAT  # GAT | GCN | GraphSAGE
baseline = False

# Dataset (AS, from pyBKT examples)
data = pd.read_csv(
    "https://github.com/CAHLR/pyBKT-examples/blob/master/data/as.csv?raw=true",
    encoding="latin",
)
print(data.shape)
data.head()

In [ ]:
# Chạy lại từ đầu: xóa cache graph + chuẩn bị thư mục ckpts
import os

for f in ("skill_graph.pickle", "skill_dict.pickle"):
    if os.path.exists(f):
        os.remove(f)

os.makedirs("ckpts", exist_ok=True)

In [ ]:
# Preprocess + build skill graph + init transformer & GNN
data_train, data_val, skill_graph, model, skill_net, optimizer = init_models(
    data,
    block_size=block_size,
    skill_embd_dim=skill_embd_dim,
    graph_net=graph_net,
    baseline=baseline,
)
print(f"Training {skill_net.tag} | train={len(data_train)} | val={len(data_val)}")
print(f"Skills: {skill_graph.number_of_nodes()} | edges: {skill_graph.number_of_edges()}")

In [ ]:
# Train end-to-end (transformer + GNN), mỗi epoch validate + lưu checkpoint vào ./ckpts
train(
    model,
    optimizer,
    skill_net,
    skill_graph,
    data_train,
    data_val,
    batch_size=batch_size,
    block_size=block_size,
    num_epochs=num_epochs,
    baseline=baseline,
)

In [ ]:
# Liệt kê checkpoints (tải về từ sidebar Output của notebook)
from pathlib import Path

ckpts = sorted(Path("ckpts").glob("*.pth"))
for p in ckpts:
    print(f"{p.stat().st_size / 1e6:8.2f} MB  {p}")
print(f"Total: {len(ckpts)} files")